<div style="text-align:center;">
  <h1 size=10>
    <b>DEEP LEARNING PROJECT</b><br>
    <b>WikiArt Painter Classification</b>
  </h1>
</div>

<h2 style="text-align:center;">
Master's in Data Science and Advanced Analytics - NOVA IMS (25/26)
</h2>

**Group XX**
- Bárbar Franco (20250388)
- Catarina Mendinhas (20250422)
- Guilherme Silva (20221910)  
- Maria Carvalho (20221953)
- Maria Miguel Fonseca (20250380)

<font color='#2f94d7' size=6>**TABLE OF CONTENTS**</font> <a class="anchor" id='toc'></a>

- [1. Approach Overview](#1)
- [2. Set up & Imports](#2)
- [3. Load Dataset](#3)
- [4. Class Weights](#4)
- [5. Data Preprocessing](#5)
- [6. Modeling](#6)
    - [6.1 Basic Model Comparison](#6_1)
    - [6.2 Optimizer Comparison](#6_2)
    - [6.3 Activation Function Comparison](#6_3)



# <font color='#2f94d7' size=6>**1. Approach Overview**</font> <a class="anchor" id="1"></a>

[Back to TOC](#toc)

# <font color='#2f94d7' size=6>**2. Set up & Imports**</font> <a class="anchor" id="2"></a>

[Back to TOC](#toc)

In [19]:
# IMPORT LIBRARIES
import os
import random
import hashlib
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
from collections import Counter
from PIL import Image
import cv2

import tensorflow as tf
import keras
from keras import layers, callbacks, optimizers
from keras.applications import EfficientNetB0

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import shutil
import seaborn as sns

# Import .py files
from source.functions import *

In [20]:
# REPRODUCIBILITY
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [21]:
# GENERAL CONFIGURATION
AUTOTUNE    = tf.data.AUTOTUNE

print('Keras  :', keras.__version__)
print('TF     :', tf.__version__)
print('GPU    :', tf.config.list_physical_devices('GPU') or 'None - CPU mode')

Keras  : 3.13.2
TF     : 2.20.0
GPU    : None - CPU mode


# <font color='#2f94d7' size=6>**3. Load Dataset**</font> <a class="anchor" id="3"></a>

[Back to TOC](#toc)

In [22]:
# DATASET CONFIGURATION
IMG_SIZE    = (512, 512)   # Initial image size (can be adapted later for modeling)
NUM_CLASSES = 23
BATCH_SIZE  = 32 

SPLIT_DIR      = Path('wikiart_split')
train_dir_path = SPLIT_DIR / 'train'
val_dir_path   = SPLIT_DIR / 'val'
test_dir_path  = SPLIT_DIR / 'test'

In [23]:
# Load datasets from pre-split folders 
# image_dataset_from_directory infers labels from subfolder names,
# guaranteeing consistent class_names order across machines.

train_ds_raw = keras.utils.image_dataset_from_directory(
    train_dir_path,
    label_mode='categorical', # one-hot encoding for multi-class classification
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    interpolation='bilinear',
    verbose=False
)

val_ds_raw = keras.utils.image_dataset_from_directory(
    val_dir_path,
    label_mode='categorical',
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    interpolation='bilinear',
    shuffle=False,
    verbose=False
)

test_ds_raw = keras.utils.image_dataset_from_directory(
    test_dir_path,
    label_mode='categorical',
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    interpolation='bilinear',
    shuffle=False,
    verbose=False
)

class_names = train_ds_raw.class_names

In [24]:
print(f"Class names: {class_names}")

# Check if labels are correctly inferred (one-hot encoded)
for images, labels in train_ds_raw.take(1):
    print(labels.shape)
    print(labels[0])

Class names: ['Albrecht_Durer', 'Boris_Kustodiev', 'Camille_Pissarro', 'Childe_Hassam', 'Claude_Monet', 'Edgar_Degas', 'Eugene_Boudin', 'Gustave_Dore', 'Ilya_Repin', 'Ivan_Aivazovsky', 'Ivan_Shishkin', 'John_Singer_Sargent', 'Marc_Chagall', 'Martiros_Saryan', 'Nicholas_Roerich', 'Pablo_Picasso', 'Paul_Cezanne', 'Pierre_Auguste_Renoir', 'Pyotr_Konchalovsky', 'Raphael_Kirchner', 'Rembrandt', 'Salvador_Dali', 'Vincent_van_Gogh']
(32, 23)
tf.Tensor([0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.], shape=(23,), dtype=float32)


# <font color='#2f94d7' size=6>**4. Class Weights**</font> <a class="anchor" id="4"></a>

[Back to TOC](#toc)

In [25]:
# Compute class weights to handle class imbalance

# Extract labels from training dataset (convert one-hot → class index)
train_labels_flat = np.concatenate([
    np.argmax(y, axis=1) for _, y in train_ds_raw
])

# Compute balanced class weights
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=train_labels_flat
)

# Convert to dictionary format required by Keras
class_weight_dict = {i: w for i, w in enumerate(class_weights_array)}

# Display classes with highest weights (rarest classes)
print("Class weights (top 5 heaviest):")
sorted_cw = sorted(class_weight_dict.items(), key=lambda x: x[1], reverse=True)

for idx, w in sorted_cw[:5]:
    print(f"  {class_names[idx]:<10}: {w:.3f}")

Class weights (top 5 heaviest):
  Salvador_Dali: 1.726
  Raphael_Kirchner: 1.599
  Ivan_Shishkin: 1.593
  Ilya_Repin: 1.531
  Childe_Hassam: 1.515


As expected, artists with fewer paintings receive higher class weights. This ensures that misclassifications on these underrepresented classes are penalized more during training, helping to reduce the impact of class imbalance.

# <font color='#2f94d7' size=6>**5. Data Preprocessing**</font> <a class="anchor" id="5"></a>

[Back to TOC](#toc)

In [26]:
# DEFINE AND APPLY DATA AUGMENTATION
base_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.03),       # ~ ±10°
    layers.RandomTranslation(0.03, 0.03),
    layers.RandomBrightness(factor=0.15),
    layers.RandomContrast(factor=0.15),
], name="base_augmentation")

train_aug = (train_ds_raw
    .shuffle(1000, seed=SEED)
    .map(lambda x, y: (base_augmentation(x, training=True), y),
         num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE))

In [27]:
# RESIZE DATASETS TO DIFFERENT SIZES
def build_resized_ds(train_ds, val_ds, image_model_size):
    def resize_fn(image, label):
        image = tf.image.resize(image, [image_model_size, image_model_size])
        return image, label

    train_ds = train_ds.map(resize_fn, num_parallel_calls=AUTOTUNE)
    val_ds = val_ds.map(resize_fn, num_parallel_calls=AUTOTUNE)

    return train_ds, val_ds

train_224, val_224 = build_resized_ds(train_aug, val_ds_raw, 224)
train_300, val_300 = build_resized_ds(train_aug, val_ds_raw, 300)
train_512, val_512 = build_resized_ds(train_aug, val_ds_raw, 512)


In [28]:
# DEF PREPROCESSING FUNCTION FOR DIFFERENT MODELS
def apply_preprocess_ds(train_resized, val_resized, preprocess_fn, batch_size=32):
    def apply_preprocess_img(image, label):
        image = preprocess_fn(image)
        return image, label

    t_ds = (train_resized
            .shuffle(len(train_resized), seed=SEED)
            .map(apply_preprocess_img, num_parallel_calls=AUTOTUNE)
            .batch(batch_size)
            .prefetch(AUTOTUNE))
    v_ds = (val_resized
            .map(apply_preprocess_img, num_parallel_calls=AUTOTUNE)
            .batch(batch_size)
            .prefetch(AUTOTUNE))
    return t_ds, v_ds

# <font color='#2f94d7' size=6>**6. Modeling**</font> <a class="anchor" id="6"></a>

[Back to TOC](#toc)

# <font color='#2f94d7' size=6>**6.1 Basic Model Comparison**</font> <a class="anchor" id="6_1"></a>

[Back to TOC](#toc)

# <font color='#2f94d7' size=6>**6.2 Optmizer Comparison**</font> <a class="anchor" id="6_2"></a>

[Back to TOC](#toc)

# <font color='#2f94d7' size=6>**6.3 Activation Function Comparison**</font> <a class="anchor" id="6_3"></a>

[Back to TOC](#toc)